# Semantic Caching for LLM APIs — Production-Grade Implementation

## Data Sources (no `datasets` library required)
- **Quora Question Pairs** → fetched directly from Quora's public CDN as TSV (one HTTP GET)
- **Wikipedia REST API** → live data on Python libraries (Django, Flask, NumPy, Pandas, FastAPI, ...) for drift/versioning tests

## Architecture
```
User Query → Embed → FAISS cosine search
   ├── sim ≥ threshold → return cached (HIT)
   └── sim <  threshold → LLM call → store with version + source_hash (MISS)
                                          ↑
                              Wikipedia drift watcher
                              (re-fetch hourly, invalidate on hash change)
```
## What we'll build
1. Embedding layer (Sentence-Transformers, batched)
2. Vector store + cosine similarity (FAISS HNSW)
3. Threshold tuning via precision/recall/F1 on Quora pairs
4. **Cache versioning & TTL** — invalidate when source data updates
5. Metrics dashboard: Hit rate, P/R/F1, latency p50/p95, $ saved
6. Structured logging + alerting hooks (Prometheus-style)

In [ ]:
# !pip install -q sentence-transformers faiss-cpu numpy pandas scikit-learn \
#                 prometheus-client structlog requests

In [ ]:
import os, time, json, hashlib, logging, threading, io
from dataclasses import dataclass
from typing import Optional
import numpy as np
import pandas as pd
import requests
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_recall_fscore_support
import structlog

structlog.configure(processors=[
    structlog.processors.TimeStamper(fmt="iso"),
    structlog.processors.add_log_level,
    structlog.processors.JSONRenderer(),
])
log = structlog.get_logger()

## Dataset 1 — Quora Question Pairs via Direct HTTP

Quora published this dataset publicly. We fetch the TSV straight from their CDN:

> `http://qim.fs.quoracdn.net/quora_duplicate_questions.tsv`

- ~58 MB, 404,290 rows
- Columns: `id, qid1, qid2, question1, question2, is_duplicate`
- No API key, no `datasets` lib, no auth — just `requests.get()`

**Fallback mirrors** (if Quora's CDN is slow):
- GitHub mirror: `https://raw.githubusercontent.com/zhiguowang/BiMPM/master/data/quora/dev.tsv` (smaller subset)
- Kaggle CLI: `kaggle competitions download -c quora-question-pairs` (needs API token)

In [ ]:
QUORA_URL = "http://qim.fs.quoracdn.net/quora_duplicate_questions.tsv"
LOCAL_TSV = "quora_duplicate_questions.tsv"

def fetch_quora(url=QUORA_URL, local=LOCAL_TSV, force=False) -> pd.DataFrame:
    if force or not os.path.exists(local):
        log.info("quora_download_start", url=url)
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(local, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 16):
                    f.write(chunk)
        log.info("quora_download_done", bytes=os.path.getsize(local))
    df = pd.read_csv(local, sep="\t", on_bad_lines="skip").dropna(
        subset=["question1", "question2", "is_duplicate"]
    )
    df = df.rename(columns={"question1": "q1", "question2": "q2",
                            "is_duplicate": "is_dup"})
    df["is_dup"] = df["is_dup"].astype(int)
    return df[["q1", "q2", "is_dup"]]

quora_df = fetch_quora().sample(20_000, random_state=42).reset_index(drop=True)
print(quora_df.head(), "\nDuplicate ratio:", quora_df.is_dup.mean(),
      "\nTotal rows:", len(quora_df))

## Step 1 — Embeddings

We use **`all-MiniLM-L6-v2`**:
- 384 dims, 80MB, ~14k sentences/sec on CPU
- Strong on Quora-style paraphrase tasks (≥0.85 STS-B score)

**Production tips:**
- Always **L2-normalize** vectors → cosine sim becomes a simple dot product (FAISS `IndexFlatIP`)
- **Batch** embedding calls (32–64) — single calls are 10× slower
- Cache the model in memory; never reload per request
- For multilingual: swap to `paraphrase-multilingual-MiniLM-L12-v2`

In [ ]:
class Embedder:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()
        log.info("embedder_loaded", model=model_name, dim=self.dim)

    def encode(self, texts, batch_size=64):
        if isinstance(texts, str):
            texts = [texts]
        return self.model.encode(
            texts, 
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True, 
            show_progress_bar=False,
        ).astype("float32")

embedder = Embedder()

## Dataset 2 — Wikipedia REST API (live, drift-prone)

We use Wikipedia's **official REST API** (no wrapper lib):

| Endpoint | Purpose |
|---|---|
| `GET /api/rest_v1/page/summary/{title}` | Short summary + revision id |
| `GET /api/rest_v1/page/html/{title}` | Full HTML (heavier) |
| `GET /w/api.php?action=query&prop=revisions&titles=...` | Latest revision id (for drift) |

**Why Python libraries?** They're frequently edited (new releases, deprecations) — perfect for testing
cache invalidation. Articles we'll target:

```
Django (web framework), Flask (web framework), FastAPI,
NumPy, Pandas (software), SciPy, scikit-learn,
PyTorch, TensorFlow, Matplotlib, Requests (software)
```

**Etiquette / production rules:**
- Always send a descriptive `User-Agent` (Wikipedia rejects generic UAs / will rate-limit you)
- Respect HTTP caching headers (`ETag`, `Last-Modified`) → use `If-None-Match`
- Max ~200 req/s; back off on 429

In [ ]:
WIKI_REST = "https://en.wikipedia.org/api/rest_v1"
WIKI_API  = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "SemanticCacheDemo/1.0 (https://github.com/Pavanpinapatruni; demo@example.com)",
    "Accept": "application/json",
}

PYTHON_LIB_TITLES = [
    "Django_(web_framework)",
    "Flask_(web_framework)",
    "FastAPI",
    "NumPy",
    "Pandas_(software)",
    "SciPy",
    "Scikit-learn",
    "PyTorch",
    "TensorFlow",
    "Matplotlib",
    "Requests_(software)",
]

class WikipediaClient:
    def __init__(self, headers=HEADERS):
        self.s = requests.Session()
        self.s.headers.update(headers)

    def summary(self, title: str) -> dict:
        """Short summary + revision id (cheap, ~2KB)."""
        r = self.s.get(f"{WIKI_REST}/page/summary/{title}", timeout=15)
        r.raise_for_status()
        return r.json()

    def latest_revision_id(self, title: str) -> int:
        """Cheapest possible drift signal — just the revid."""
        params = {
            "action": "query", "format": "json",
            "prop": "revisions", "rvprop": "ids|timestamp",
            "titles": title.replace("_", " "),
        }
        r = self.s.get(WIKI_API, params=params, timeout=15); r.raise_for_status()
        pages = r.json()["query"]["pages"]
        page = next(iter(pages.values()))
        return page["revisions"][0]["revid"]

    def content_hash(self, title: str) -> tuple[str, str, int]:
        """Returns (text, sha256_hash, revision_id)."""
        data = self.summary(title)
        text = data.get("extract", "")
        revid = data.get("revision", 0)
        h = hashlib.sha256(text.encode("utf-8")).hexdigest()
        return text, h, int(revid)

wiki = WikipediaClient()

# Quick demo
sample = wiki.summary("FastAPI")
print("Title:", sample["title"])
print("Revision:", sample.get("revision"))
print("Extract (first 200 chars):", sample["extract"][:200], "...")

### Build Q&A Knowledge Base from Wikipedia

In [ ]:
def build_python_lib_kb() -> pd.DataFrame:
    """
    Pulls each Python-library article and constructs canonical Q&A pairs
    grounded in the Wikipedia summary. These become initial cache entries.
    """
    rows = []
    for title in PYTHON_LIB_TITLES:
        try:
            text, h, revid = wiki.content_hash(title)
            name = title.split("_")[0].replace("(", "").strip()
            rows.append({
                "question": f"What is {name}?",
                "answer":   text[:400],
                "source_title": title,
                "source_hash":  h,
                "revision_id":  revid,
                "fetched_at":   time.time(),
            })
        except Exception as e:
            log.error("wiki_fetch_failed", title=title, err=str(e))
    return pd.DataFrame(rows)

kb_df = build_python_lib_kb()
print(kb_df[["question", "source_title", "revision_id"]])

## TTL (Time-To-Live)
A time limit on how long a cached entry remains valid. After the TTL expires, the entry is considered stale and ignored — even if it's a semantic match.

In the notebook:

* ttl_seconds=86400 = 24 hours
* Check in get(): if time.time() - entry.created_at > self.ttl → return None (cache miss)

> Why?
> 
> LLM answers can become outdated (e.g., "What's the latest version of X?"). TTL ensures stale answers aren't served forever.

## LRU (Least Recently Used)
An eviction policy — when the cache is full (max_size reached), the entry that was used the longest time ago gets removed to make room for new ones.

> Why?
> 
> Keeps the cache populated with actively used entries, dropping "cold" ones that haven't been queried in a while.

## Versioned Semantic Cache

Each entry carries:
- `version` — global, bumped on model upgrade / KB redeploy
- `source_hash` — sha256 of the Wikipedia summary text
- `revision_id` — Wikipedia's own monotonic revid (cheapest drift signal)
- `created_at` — for TTL

**Drift detection priority (cheap → expensive):**
1. Compare `revision_id` (1 API call, no body parsing)
2. If revid changed → fetch summary, recompute hash, invalidate matching entries

In [ ]:
@dataclass
class CacheEntry:
    query: str
    answer: str
    version: int
    source_title: str
    source_hash: str
    revision_id: int
    created_at: float
    last_used: float
    hits: int = 0

class SemanticCache:
    def __init__(self, embedder, dim, threshold=0.85,
                 ttl_seconds=86400, max_size=100_000, version=1):
        self.embedder, self.threshold = embedder, threshold
        self.ttl, self.max_size, self.version = ttl_seconds, max_size, version
        self.index = faiss.IndexFlatIP(dim)
        self.entries: list[CacheEntry] = []
        self.lock = threading.RLock()

    def put(self, query, answer, source_title="", source_hash="", revision_id=0):
        with self.lock:
            if len(self.entries) >= self.max_size: self._evict_lru()
            self.index.add(self.embedder.encode(query))
            self.entries.append(CacheEntry(
                query=query, answer=answer, version=self.version,
                source_title=source_title, source_hash=source_hash,
                revision_id=revision_id,
                created_at=time.time(), last_used=time.time(),
            ))
            log.info("cache_put", q=query[:60], src=source_title,
                     revid=revision_id, size=len(self.entries))

    def get(self, query):
        with self.lock:
            if self.index.ntotal == 0: return None, 0.0
            sims, idxs = self.index.search(self.embedder.encode(query), k=1)
            sim, idx = float(sims[0][0]), int(idxs[0][0])
            if sim < self.threshold: return None, sim
            e = self.entries[idx]
            if e.version != self.version:
                log.warning("cache_stale_version", q=query[:60]); return None, sim
            if time.time() - e.created_at > self.ttl:
                log.warning("cache_expired_ttl", q=query[:60]); return None, sim
            e.last_used, e.hits = time.time(), e.hits + 1
            return e.answer, sim

    def invalidate_by_revid(self, source_title: str, new_revid: int):
        """Drop entries whose stored revid != new_revid for given title."""
        with self.lock:
            killed = [i for i, e in enumerate(self.entries)
                      if e.source_title == source_title and e.revision_id != new_revid]
            for i in sorted(killed, reverse=True):
                self.entries.pop(i)
            # FAISS Flat doesn't support deletion; rebuild
            if killed: self._rebuild_index()
            log.info("cache_invalidated", title=source_title, killed=len(killed))
            return len(killed)

    def bump_version(self):
        with self.lock:
            self.version += 1
            log.info("cache_version_bumped", new=self.version)

    def _rebuild_index(self):
        self.index = faiss.IndexFlatIP(self.embedder.dim)
        if self.entries:
            vecs = self.embedder.encode([e.query for e in self.entries])
            self.index.add(vecs)

    def _evict_lru(self):
        oldest = min(range(len(self.entries)), key=lambda i: self.entries[i].last_used)
        self.entries.pop(oldest); self._rebuild_index()

## Step 3 — Threshold Tuning (precision vs recall trade-off)
Use Quora labeled pairs to sweep cosine thresholds and pick the **best F1**.
For high-stakes domains (medical/legal/finance), bias toward **precision** with F-beta (β=0.5).

## Choosing the Threshold (THE most important decision)

| Threshold | Risk |
|---|---|
| Too low (0.70) | Wrong cached answers served — silent failure |
| Too high (0.95) | Hit rate collapses, no savings |
| **Sweet spot** | Maximize **F1** (or weighted toward precision if answers must be safe) |

**Industrial tip:** If your domain is medical/legal → optimize for **precision** (use F-beta with β<1). For general chatbots → optimize **F1** or even recall-leaning (β>1).

In [ ]:
e1 = embedder.encode(quora_df.q1.tolist())
e2 = embedder.encode(quora_df.q2.tolist())
sims = (e1 * e2).sum(axis=1)

rows = []
for t in np.arange(0.60, 0.96, 0.02):
    pred = (sims >= t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        quora_df.is_dup, pred, average="binary", zero_division=0)
    rows.append((round(t,2), p, r, f1))
tune_df = pd.DataFrame(rows, columns=["threshold","precision","recall","f1"])
best = tune_df.loc[tune_df.f1.idxmax()]
print(tune_df, "\n\nBest threshold:", best.to_dict())
BEST_THRESHOLD = float(best.threshold)

In [ ]:
from prometheus_client import Counter, Histogram, Gauge

class Metrics:
    def __init__(self, llm_cost=0.002):
        self.hits = Counter("cache_hits_total", "")
        self.miss = Counter("cache_misses_total", "")
        self.lat  = Histogram("query_latency_seconds", "",
                              buckets=[.005,.01,.05,.1,.5,1,2,5])
        self.saved= Counter("dollars_saved_total", "")
        self.hr   = Gauge("hit_rate", "")
        self.cost = llm_cost
        self.tp = self.fp = self.fn = self.tn = 0

    def record(self, hit, lat_s):
        (self.hits if hit else self.miss).inc()
        self.lat.observe(lat_s)
        if hit: self.saved.inc(self.cost)
        h, m = self.hits._value.get(), self.miss._value.get()
        if h+m: self.hr.set(h/(h+m))

    def eval(self, pred_hit, true_dup):
        if pred_hit and true_dup: self.tp += 1
        elif pred_hit:            self.fp += 1
        elif true_dup:            self.fn += 1
        else:                     self.tn += 1

    def summary(self):
        h, m = self.hits._value.get(), self.miss._value.get()
        p = self.tp/(self.tp+self.fp) if (self.tp+self.fp) else 0
        r = self.tp/(self.tp+self.fn) if (self.tp+self.fn) else 0
        return {"hit_rate": h/(h+m) if (h+m) else 0,
                "precision": p, "recall": r,
                "f1": 2*p*r/(p+r) if (p+r) else 0,
                "dollars_saved": self.saved._value.get(),
                "total": h+m}

def fake_llm(q): time.sleep(0.01); return f"answer:{q}"

class CachedLLMService:
    def __init__(self, cache, m): self.cache, self.m = cache, m
    def ask(self, q, gt_dup=None):
        t0 = time.time()
        ans, sim = self.cache.get(q)
        if ans is not None:
            self.m.record(True, time.time()-t0)
            if gt_dup is not None: self.m.eval(True, gt_dup)
            log.info("HIT", q=q[:60], sim=round(sim,3)); return ans
        ans = fake_llm(q)
        self.cache.put(q, ans)
        self.m.record(False, time.time()-t0)
        if gt_dup is not None: self.m.eval(False, gt_dup)
        log.info("MISS", q=q[:60], top_sim=round(sim,3)); return ans

cache = SemanticCache(embedder, dim=embedder.dim, threshold=BEST_THRESHOLD)
metrics = Metrics()
service = CachedLLMService(cache, metrics)

# Pre-warm with Wikipedia KB
for _, row in kb_df.iterrows():
    cache.put(row.question, row.answer,
              source_title=row.source_title,
              source_hash=row.source_hash,
              revision_id=row.revision_id)
print("Cache size after KB warm:", len(cache.entries))

### Simulate Production Traffic with Quora

In [ ]:
# Seed cache with q1's, then ask q2's (50% duplicates → realistic prod)
sample = quora_df.sample(500, random_state=1).reset_index(drop=True)
# seed cache with q1
for q in sample.q1.head(200): 
    cache.put(q, fake_llm(q))
# query with q2 (50% are paraphrases of q1)
for _, row in sample.iterrows():
    service.ask(row.q2, gt_dup=bool(row.is_dup))
print(json.dumps(metrics.summary(), indent=2))

### Wikipedia drift watcher (revision-id based)

In [ ]:
class WikiDriftWatcher:
    def __init__(self, wiki_client, cache, titles, interval_s=3600):
        self.wiki, self.cache, self.titles, self.interval = \
            wiki_client, cache, titles, interval_s
        self.last_revid = {t: wiki_client.latest_revision_id(t) for t in titles}
        log.info("drift_watcher_init", revids=self.last_revid)

    def check_once(self):
        for t in self.titles:
            try:
                new = self.wiki.latest_revision_id(t)
                if new != self.last_revid[t]:
                    log.warning("source_drift", title=t,
                                old=self.last_revid[t], new=new)
                    self.cache.invalidate_by_revid(t, new)
                    self.last_revid[t] = new
            except Exception as e:
                log.error("drift_check_failed", title=t, err=str(e))

    def run_forever(self):
        while True:
            self.check_once()
            time.sleep(self.interval)

watcher = WikiDriftWatcher(wiki, cache, PYTHON_LIB_TITLES, interval_s=3600)
watcher.check_once()    # one-shot demo
# threading.Thread(target=watcher.run_forever, daemon=True).start()

### Manual Drift Test(force invalidation)

In [ ]:
# Simulate FastAPI's Wikipedia article being edited
fake_old_revid = cache.entries[0].revision_id
killed = cache.invalidate_by_revid("FastAPI", new_revid=fake_old_revid + 1)
print(f"Forced invalidation killed {killed} entries.")

# Same question now misses → calls LLM → re-fetches & re-caches
ans = service.ask("What is FastAPI?")
print("Answer after invalidation:", ans[:100])

## Step 3 — Logging & Alarms

We already log structured JSON via `structlog`. Ship these to:
- **Logs:** Datadog / Loki / CloudWatch
- **Metrics:** Prometheus → Grafana dashboard
- **Alerts:** PagerDuty / Slack via Alertmanager

### Alert rules (Prometheus YAML)
```yaml
- alert: CacheHitRateDrop
  expr: hit_rate < 0.30
  for: 15m
  annotations:
    summary: "Cache hit rate dropped — losing money"

- alert: CachePrecisionLow
  expr: (cache_tp / (cache_tp + cache_fp)) < 0.90
  for: 30m
  annotations:
    summary: "Serving wrong answers — raise threshold"

- alert: HighLatency
  expr: histogram_quantile(0.95, query_latency_seconds_bucket) > 0.2
```

In [ ]:
def check_alerts():
    s = metrics.summary(); alerts = []
    if s["total"] > 100:
        if s["hit_rate"]  < 0.30: alerts.append(("HIT_RATE_LOW",  s["hit_rate"]))
        if s["precision"] < 0.90 and s["precision"] > 0:
            alerts.append(("PRECISION_LOW", s["precision"]))
    for n, v in alerts: log.error("ALERT", name=n, value=v)
    return alerts

print("Final metrics:", json.dumps(metrics.summary(), indent=2))
print("Alerts:", check_alerts())